# LeetCode 3311 — Construct 2D Grid Matching Graph Layout bằng Python

- Link: https://leetcode.com/problems/construct-2d-grid-matching-graph-layout/
- Độ khó: **Hard**
- Chủ đề: **Nhận dạng đồ thị lưới qua bậc** (góc bậc 2, cạnh bậc 3, trong bậc 4), dựng **hàng đầu** từ một góc, rồi mỗi hàng sau là **hàng xóm chưa dùng duy nhất** của hàng trên; cách khác: **tọa độ từ khoảng cách BFS** tới hai góc

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def constructGridLayout(self, n: int, edges: List[List[int]]) -> List[List[int]]:
```

## 1. Mục tiêu học

1. Đọc cấu trúc lưới `r × c` qua **bậc**: với `r, c ≥ 2`, góc bậc 2, biên bậc 3, trong bậc 4; với một hàng (`r = 1`), hai đầu bậc 1.
2. Dựng hàng đầu tiên: xuất phát từ một góc, đi theo các nút **biên** (bậc `< 4`) cho tới góc kế tiếp; xử lý riêng lưới **2 hàng** (mọi nút đều bậc ≤ 3).
3. Hàng `k + 1`: mỗi nút ở hàng `k` có **đúng một** hàng xóm chưa dùng — chính là nút ngay dưới.
4. Cách thứ hai: tọa độ `(i, j)` suy ra từ khoảng cách BFS tới góc `(0, 0)` và góc `(0, c − 1)`.
5. Vì có nhiều đáp án, test bằng **hàm kiểm tra** (tập cạnh của lưới = tập cạnh đề cho).

## 2. Đề bài & ví dụ

Cho đồ thị vô hướng `n` nút với danh sách cạnh `edges`. Dựng một lưới 2D chứa mỗi nút `0..n−1` đúng một lần sao cho hai nút **kề nhau trong lưới** (ngang/dọc) **khi và chỉ khi** có cạnh giữa chúng. Đề bảo đảm có lời giải; trả về bất kỳ lưới hợp lệ nào.

**Ví dụ 1:** `n = 4`, `edges = [[0,1],[0,2],[1,3],[2,3]]` → `[[3,1],[2,0]]`.
**Ví dụ 2:** `n = 5`, `edges = [[0,1],[1,3],[2,3],[2,4]]` → `[[4,2,3,1,0]]`.
**Ví dụ 3:** `n = 9`, … → `[[8,6,3],[7,4,2],[1,0,5]]`.

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`; `1 ≤ len(edges) ≤ 10^5`; cạnh phân biệt.

In [1]:
from typing import List
from collections import deque
from itertools import permutations

examples = [
    (4, [[0, 1], [0, 2], [1, 3], [2, 3]]),
    (5, [[0, 1], [1, 3], [2, 3], [2, 4]]),
    (9, [[0, 1], [0, 4], [0, 5], [1, 7], [2, 3], [2, 4], [2, 5], [3, 6], [4, 6], [4, 7], [6, 8], [7, 8]]),
]

def grid_edges(grid):
    r, c = len(grid), len(grid[0])
    out = set()
    for i in range(r):
        for j in range(c):
            if j + 1 < c:
                out.add(frozenset((grid[i][j], grid[i][j + 1])))
            if i + 1 < r:
                out.add(frozenset((grid[i][j], grid[i + 1][j])))
    return out

def is_valid(n, edges, grid):
    if not grid or any(len(row) != len(grid[0]) for row in grid):
        return False
    cells = [v for row in grid for v in row]
    if sorted(cells) != list(range(n)):
        return False
    return grid_edges(grid) == {frozenset(e) for e in edges}

print('đáp án đề hợp lệ:', [is_valid(4, examples[0][1], [[3, 1], [2, 0]]),
                            is_valid(5, examples[1][1], [[4, 2, 3, 1, 0]]),
                            is_valid(9, examples[2][1], [[8, 6, 3], [7, 4, 2], [1, 0, 5]])])

đáp án đề hợp lệ: [True, True, True]


## 3. Kiến thức nền

### 3.1. Bậc trong lưới

In [2]:
def degree_map(r, c):
    return [[(i > 0) + (i < r - 1) + (j > 0) + (j < c - 1) for j in range(c)] for i in range(r)]

for r, c in [(1, 5), (2, 4), (3, 4)]:
    print(f"{r}×{c}:", degree_map(r, c))

1×5: [[1, 2, 2, 2, 1]]
2×4: [[2, 3, 3, 2], [2, 3, 3, 2]]
3×4: [[2, 3, 3, 2], [3, 4, 4, 3], [2, 3, 3, 2]]


- `1 × c`: hai đầu bậc 1 (nếu `c ≥ 2`), giữa bậc 2 — một **đường thẳng**.
- `2 × c`: góc bậc 2, còn lại bậc 3 — **không có nút bậc 4**, nên "đi theo biên" bị mơ hồ.
- `r, c ≥ 3`: hàng biên gồm góc (2) và các nút bậc 3; hàng thứ hai có nút bậc 4.

### 3.2. Hàng sau xác định duy nhất

Nếu đã biết hàng `k` (và mọi hàng trên đã đánh dấu), nút `(k, j)` có các hàng xóm `(k−1, j)`, `(k, j±1)` đã dùng, và `(k+1, j)` chưa dùng ⇒ hàng xóm chưa dùng **duy nhất** là nút ngay dưới.

### 3.3. Khoảng cách đồ thị = khoảng cách Manhattan

Trong đồ thị lưới, đường ngắn nhất giữa `(i1, j1)` và `(i2, j2)` dài đúng `|i1 − i2| + |j1 − j2|` (mỗi cạnh đổi đúng một tọa độ 1 đơn vị, và luôn đi được "đơn điệu"). Đây là cơ sở của cách 3. Kiểm tra nhanh bằng BFS:

In [3]:
def _bfs(adj, s):
    d = {s: 0}; q = [s]
    for u in q:
        for v in adj[u]:
            if v not in d:
                d[v] = d[u] + 1; q.append(v)
    return d

R_, C_ = 4, 6
adj_ = {(i, j): [(i + di, j + dj) for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1))
                 if 0 <= i + di < R_ and 0 <= j + dj < C_] for i in range(R_) for j in range(C_)}
ok_ = all(_bfs(adj_, a)[b] == abs(a[0] - b[0]) + abs(a[1] - b[1]) for a in adj_ for b in adj_)
print('Lưới 4×6: khoảng cách BFS = Manhattan cho mọi cặp ô:', ok_)

Lưới 4×6: khoảng cách BFS = Manhattan cho mọi cặp ô: True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Lưới `3 × 3` ví dụ 3: bậc của các nút cho biết góc (bậc 2) là `{8, 3, 1, 5}`, tâm (bậc 4) là `4`. Từ góc `8`: hai hàng xóm `6`, `7` đều bậc 3 — chọn `6`; từ `6` đi tiếp sang hàng xóm bậc `< 4` chưa dùng: `3` (góc) ⇒ hàng đầu `[8, 6, 3]`. Hàng hai: dưới `8` là `7`, dưới `6` là `4`, dưới `3` là `2`. Hàng ba: `1, 0, 5` ✓.

**Lưới 2 hàng:** từ góc, một hàng xóm bậc 3 (dọc theo cạnh dài) và một hàng xóm bậc 2 (góc cùng cột). Nếu chọn hàng xóm bậc 2, hàng đầu chỉ dài 2 — ta dựng lưới **chuyển vị** `c × 2`, vẫn hợp lệ và không còn mơ hồ. Quy tắc: **chọn hàng xóm có bậc nhỏ nhất**.

## 5. Cách 1 — Brute Force: thử mọi hoán vị vào mọi hình dạng

In [4]:
def brute(n, edges):
    target = {frozenset(e) for e in edges}
    for r in range(1, n + 1):
        if n % r:
            continue
        c = n // r
        for perm in permutations(range(n)):
            grid = [list(perm[i * c:(i + 1) * c]) for i in range(r)]
            if grid_edges(grid) == target:
                return grid
    return []

for n, edges in examples[:2]:
    g = brute(n, edges)
    print(g, '| hợp lệ:', is_valid(n, edges, g))

[[0, 1], [2, 3]] | hợp lệ: True
[[0, 1, 3, 2, 4]] | hợp lệ: True


**Độ phức tạp:** `O(n! · n)` — chỉ `n ≤ 8`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính bậc; `x` = nút bậc nhỏ nhất.
2. **Nếu `deg[x] = 1`** ⇒ lưới một hàng: đi dọc đường thẳng từ `x`.
3. **Ngược lại** (`deg[x] = 2`, là góc): hàng đầu `row = [x, y]` với `y` = hàng xóm của `x` có **bậc nhỏ nhất**; trong khi `deg[row[-1]] ≠ 2` (chưa tới góc): thêm hàng xóm chưa dùng có bậc `< 4` (với lưới ≥ 3 hàng là duy nhất; với 2 hàng thì nhánh này không chạy vì `y` đã là góc).
4. Các hàng sau: với mỗi nút hàng trên, lấy hàng xóm chưa dùng duy nhất. Lặp tới khi dùng hết `n` nút.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Dựng hàng đầu + mở rộng xuống (lời giải chính)

In [5]:
def row_by_row(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    deg = [len(a) for a in g]
    x = min(range(n), key=lambda v: deg[v])
    used = [False] * n
    if deg[x] == 1:                                   # một hàng
        row, used[x] = [x], True
        while len(row) < n:
            nxt = next(v for v in g[row[-1]] if not used[v])
            used[nxt] = True; row.append(nxt)
        return [row]
    y = min(g[x], key=lambda v: deg[v])
    row = [x, y]; used[x] = used[y] = True
    while deg[row[-1]] != 2:
        nxt = next(v for v in g[row[-1]] if not used[v] and deg[v] < 4)
        used[nxt] = True; row.append(nxt)
    grid = [row]
    placed = len(row)
    while placed < n:
        new = []
        for u in grid[-1]:
            v = next(w for w in g[u] if not used[w])
            used[v] = True; new.append(v)
        grid.append(new); placed += len(new)
    return grid

for n, edges in examples:
    gr = row_by_row(n, edges)
    print(gr, '| hợp lệ:', is_valid(n, edges, gr))

[[0, 1], [2, 3]] | hợp lệ: True
[[0, 1, 3, 2, 4]] | hợp lệ: True
[[1, 0, 5], [7, 4, 2], [8, 6, 3]] | hợp lệ: True


### 7.2. Cách 3 — Tọa độ từ khoảng cách BFS tới hai góc

Trong lưới, khoảng cách đồ thị = khoảng cách Manhattan. Gọi `A` = góc `(0, 0)`, `B` = góc `(0, c − 1)` (góc **gần `A` nhất** — nằm trên cạnh ngắn hơn, ta coi cạnh đó là "hàng"). Với nút `(i, j)`: `dA = i + j`, `dB = i + (c − 1 − j)` ⇒ `i = (dA + dB − (c − 1)) / 2`, `j = (dA − dB + c − 1) / 2`.

In [6]:
def bfs(g, s):
    d = [-1] * len(g); d[s] = 0; q = deque([s])
    while q:
        u = q.popleft()
        for v in g[u]:
            if d[v] < 0:
                d[v] = d[u] + 1; q.append(v)
    return d

def by_distances(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    deg = [len(a) for a in g]
    mind = min(deg)
    corners = [v for v in range(n) if deg[v] == mind]
    A = corners[0]
    dA = bfs(g, A)
    B = min((v for v in corners if v != A), key=lambda v: dA[v])     # góc gần A nhất
    dB = bfs(g, B)
    c = dA[B] + 1
    r = n // c
    grid = [[-1] * c for _ in range(r)]
    for v in range(n):
        i = (dA[v] + dB[v] - (c - 1)) // 2
        j = (dA[v] - dB[v] + c - 1) // 2
        grid[i][j] = v
    return grid

for n, edges in examples:
    gr = by_distances(n, edges)
    print(gr, '| hợp lệ:', is_valid(n, edges, gr))

[[0, 1], [2, 3]] | hợp lệ: True
[[0, 1, 3, 2, 4]] | hợp lệ: True
[[1, 0, 5], [7, 4, 2], [8, 6, 3]] | hợp lệ: True


Với lưới một hàng, hai "góc" là hai đầu bậc 1 và `c = n`, `r = 1` — công thức vẫn đúng.

## 8. TRACE từng bước

Trace cách 2 trên một lưới `3 × 4` đã xáo nhãn.

In [7]:
import random
def make_grid_graph(r, c, rng):
    lab = list(range(r * c)); rng.shuffle(lab)
    grid = [[lab[i * c + j] for j in range(c)] for i in range(r)]
    edges = [list(e) for e in grid_edges(grid)]
    rng.shuffle(edges)
    return r * c, edges, grid

def trace(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    deg = [len(a) for a in g]
    x = min(range(n), key=lambda v: deg[v])
    print('bậc:', deg, '| góc chọn:', x, '| hàng xóm & bậc:', [(v, deg[v]) for v in g[x]])
    gr = row_by_row(n, edges)
    for k, row in enumerate(gr):
        print(f'  hàng {k}:', row, '| bậc:', [deg[v] for v in row])
    print('hợp lệ:', is_valid(n, edges, gr))

n, edges, _ = make_grid_graph(3, 4, random.Random(7))
trace(n, edges)

bậc: [2, 3, 3, 3, 4, 2, 3, 2, 3, 4, 2, 3] | góc chọn: 0 | hàng xóm & bậc: [(6, 3), (8, 3)]
  hàng 0: [0, 6, 2, 5] | bậc: [2, 3, 3, 2]
  hàng 1: [8, 4, 9, 1] | bậc: [3, 4, 4, 3]
  hàng 2: [7, 11, 3, 10] | bậc: [2, 3, 3, 2]
hợp lệ: True


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def constructGridLayout(self, n: int, edges: List[List[int]]) -> List[List[int]]:
        g = [[] for _ in range(n)]
        for u, v in edges:
            g[u].append(v)
            g[v].append(u)
        deg = [len(a) for a in g]
        x = min(range(n), key=deg.__getitem__)
        used = [False] * n
        used[x] = True
        if deg[x] == 1:
            row = [x]
            while len(row) < n:
                for v in g[row[-1]]:
                    if not used[v]:
                        used[v] = True
                        row.append(v)
                        break
            return [row]
        y = min(g[x], key=deg.__getitem__)
        used[y] = True
        row = [x, y]
        while deg[row[-1]] != 2:
            for v in g[row[-1]]:
                if not used[v] and deg[v] < 4:
                    used[v] = True
                    row.append(v)
                    break
        grid = [row]
        placed = len(row)
        while placed < n:
            new = []
            for u in grid[-1]:
                for v in g[u]:
                    if not used[v]:
                        used[v] = True
                        new.append(v)
                        break
            grid.append(new)
            placed += len(new)
        return grid

for n, edges in examples:
    gr = Solution().constructGridLayout(n, edges)
    print(gr, '| hợp lệ:', is_valid(n, edges, gr))

[[0, 1], [2, 3]] | hợp lệ: True
[[0, 1, 3, 2, 4]] | hợp lệ: True
[[1, 0, 5], [7, 4, 2], [8, 6, 3]] | hợp lệ: True


## 10. Vì sao thuật toán đúng

1. **Một hàng ⇔ có nút bậc 1:** lưới có `r, c ≥ 2` thì mọi nút bậc `≥ 2`; lưới `1 × c` (`c ≥ 2`) có hai đầu bậc 1. Khi đó đồ thị là đường thẳng, đi dọc là đúng.
2. **Hàng đầu (`r, c ≥ 2`):** `x` là góc. Nếu một hàng xóm của `x` có bậc 2 thì nó là góc khác và cạnh lưới tương ứng dài 2 (lưới `2 × c` hoặc `r × 2`); chọn nó làm hàng đầu dài 2 — hợp lệ. Nếu cả hai hàng xóm bậc 3, cả hai cạnh dài `≥ 3`; đi theo một hướng, mỗi nút biên (bậc 3) có đúng một hàng xóm chưa dùng bậc `< 4` (nút kế tiếp trên biên), nút còn lại ở hàng 2 có bậc 4 (vì cạnh kia dài `≥ 3`) — dừng khi gặp góc.
3. **Hàng sau:** mục 3.2 — duy nhất, đúng nút ngay dưới.
4. Do dựng đúng một cách đặt các nút vào lưới đúng hình dạng (một trong các phép đối xứng của lưới gốc), các cạnh khớp hoàn toàn.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n! · n)` | `O(n)` |
| 2. Hàng đầu + mở rộng | `O(n + E)` | `O(n + E)` |
| 3. Tọa độ từ BFS | `O(n + E)` | `O(n + E)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên (kiểm bằng hàm hợp lệ)

In [9]:
impls = {'row_by_row': row_by_row, 'by_distances': by_distances,
         'submit': lambda n, e: Solution().constructGridLayout(n, e)}
rng = random.Random(1)
edge_cases = [make_grid_graph(r, c, rng)[:2] for r, c in [(1, 2), (2, 1), (2, 2), (2, 5), (5, 2), (1, 7), (3, 3), (4, 3)]]
for n, edges in examples + edge_cases:
    for name, f in impls.items():
        assert is_valid(n, edges, f(n, edges)), (name, n, edges)
for n, edges in examples[:2] + edge_cases[:3]:
    assert is_valid(n, edges, brute(n, edges))
print('Ví dụ đề và ca biên: mọi lưới hợp lệ.')

Ví dụ đề và ca biên: mọi lưới hợp lệ.


### 12.2. Lưới ngẫu nhiên (xáo nhãn, xáo cạnh)

In [10]:
rng = random.Random(3311)
cnt = 0
for _ in range(500):
    r, c = rng.randint(1, 8), rng.randint(1, 8)
    if r * c < 2:
        continue
    n, edges, _ = make_grid_graph(r, c, rng)
    for name, f in impls.items():
        assert is_valid(n, edges, f(n, edges)), (name, r, c)
    cnt += 1
print('Mọi cách hợp lệ trên', cnt, 'lưới ngẫu nhiên.')

Mọi cách hợp lệ trên 491 lưới ngẫu nhiên.


### 12.3. Ca lớn `n ≈ 5·10^4`

In [11]:
import time
for r, c in [(1, 50000), (2, 25000), (223, 224), (25000, 2), (100, 500)]:
    n, edges, _ = make_grid_graph(r, c, rng)
    t = time.time(); gr = Solution().constructGridLayout(n, edges); t1 = time.time() - t
    t = time.time(); gr2 = by_distances(n, edges); t2 = time.time() - t
    assert is_valid(n, edges, gr) and is_valid(n, edges, gr2)
    print(f"{r:5d} × {c:5d}: kết quả {len(gr)} × {len(gr[0])} | nộp {t1:.2f}s | BFS tọa độ {t2:.2f}s")

    1 × 50000: kết quả 1 × 50000 | nộp 0.05s | BFS tọa độ 0.13s


    2 × 25000: kết quả 25000 × 2 | nộp 0.04s | BFS tọa độ 0.07s


  223 ×   224: kết quả 224 × 223 | nộp 0.12s | BFS tọa độ 0.11s


25000 ×     2: kết quả 25000 × 2 | nộp 0.06s | BFS tọa độ 0.11s


  100 ×   500: kết quả 500 × 100 | nộp 0.04s | BFS tọa độ 0.11s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Hàng đầu: đi sang hàng xóm chưa dùng **bất kỳ** (không lọc bậc `< 4`)

In [12]:
def wrong_any_neighbor(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    deg = [len(a) for a in g]
    x = min(range(n), key=lambda v: deg[v]); used = [False] * n; used[x] = True
    y = min(g[x], key=lambda v: deg[v]); used[y] = True
    row = [x, y]
    while deg[row[-1]] != 2:
        cand = [v for v in g[row[-1]] if not used[v]]
        if not cand:
            return [row]
        nxt = max(cand, key=lambda v: deg[v])                # SAI: có thể đi vào trong lưới (bậc 4)
        used[nxt] = True; row.append(nxt)
        if len(row) > n:
            break
    grid = [row]; placed = len(row)
    try:
        while placed < n:
            new = []
            for u in grid[-1]:
                v = next(w for w in g[u] if not used[w]); used[v] = True; new.append(v)
            grid.append(new); placed += len(new)
    except StopIteration:
        return grid
    return grid

n, edges, _ = make_grid_graph(3, 3, random.Random(2))
out = wrong_any_neighbor(n, edges)
print('sai :', out, '| hợp lệ:', is_valid(n, edges, out))

sai : [[0, 2, 7, 1, 8]] | hợp lệ: False


**Vì sao sai:** từ nút biên bậc 3, có hai hàng xóm chưa dùng: nút kế tiếp trên biên (bậc 3) và nút bên trong (bậc 4). Chọn nhầm nút trong thì "hàng" đầu không còn thẳng.

### Lỗi 2 — Lưới 2 hàng: chọn hàng xóm bậc **lớn** của góc

In [13]:
def wrong_two_rows(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    deg = [len(a) for a in g]
    x = min(range(n), key=lambda v: deg[v]); used = [False] * n; used[x] = True
    y = max(g[x], key=lambda v: deg[v]); used[y] = True        # SAI với lưới 2 × c
    row = [x, y]
    while deg[row[-1]] != 2:
        nxt = next((v for v in g[row[-1]] if not used[v] and deg[v] < 4), None)   # mơ hồ: mọi nút đều bậc <= 3
        if nxt is None:
            return [row]
        used[nxt] = True; row.append(nxt)
    grid = [row]; placed = len(row)
    while placed < n:
        new = []
        for u in grid[-1]:
            cand = [w for w in g[u] if not used[w]]
            if not cand:
                return grid
            used[cand[0]] = True; new.append(cand[0])
        grid.append(new); placed += len(new)
    return grid

bad_found = None
for seed in range(50):
    n, edges, _ = make_grid_graph(2, 5, random.Random(seed))
    out = wrong_two_rows(n, edges)
    if not is_valid(n, edges, out):
        bad_found = (seed, out); break
print('seed', bad_found[0], '-> sai:', bad_found[1], '| hợp lệ:', False)

seed 0 -> sai: [[3, 5, 9, 6]] | hợp lệ: False


**Vì sao sai:** trong lưới `2 × c`, nút biên có hai hàng xóm chưa dùng đều bậc 3 (bên phải và bên dưới) — bộ lọc `deg < 4` không phân biệt được. Chọn hàng xóm bậc 2 của góc (đi theo cạnh ngắn) tránh hoàn toàn mơ hồ.

### Lỗi 3 — Quên trường hợp một hàng (bậc 1)

In [14]:
def wrong_no_single_row(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    deg = [len(a) for a in g]
    x = min(range(n), key=lambda v: deg[v]); used = [False] * n; used[x] = True
    y = min(g[x], key=lambda v: deg[v]); used[y] = True
    row = [x, y]
    while deg[row[-1]] != 2:                                   # SAI: trên đường thẳng, nút giữa bậc 2 => dừng sớm
        nxt = next((v for v in g[row[-1]] if not used[v] and deg[v] < 4), None)
        if nxt is None:
            break
        used[nxt] = True; row.append(nxt)
    return [row]

n, edges = examples[1]
out = wrong_no_single_row(n, edges)
print('sai :', out, '| hợp lệ:', is_valid(n, edges, out), '| đúng:', Solution().constructGridLayout(n, edges))

sai : [[0, 1]] | hợp lệ: False | đúng: [[0, 1, 3, 2, 4]]


**Vì sao sai:** với lưới `1 × c`, bậc 2 là nút **giữa** chứ không phải góc; điều kiện dừng "gặp bậc 2" làm hàng chỉ dài 2.

### Lỗi 4 — Hàng sau: không đánh dấu `used` (lấy nhầm nút phía trên)

In [15]:
def wrong_no_mark(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    first = Solution().constructGridLayout(n, edges)[0]
    used = set(first)
    grid = [first]
    while sum(map(len, grid)) < n:
        new = []
        for u in grid[-1]:
            v = next((w for w in g[u] if w not in used), u)
            new.append(v)                                       # SAI: quên used.add(v)
        grid.append(new)
        if len(grid) > n:
            break
    return grid

n, edges, _ = make_grid_graph(4, 3, random.Random(5))
out = wrong_no_mark(n, edges)
print('sai :', out, '| hợp lệ:', is_valid(n, edges, out))

sai : [[5, 4, 9], [6, 0, 8], [0, 6, 0], [6, 0, 6]] | hợp lệ: False


**Vì sao sai:** ở hàng thứ ba, nút hàng hai có hàng xóm "chưa dùng" gồm cả nút hàng hai bên cạnh (vì chưa đánh dấu) — có thể chọn nhầm sang ngang.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Khôi phục cấu trúc quen thuộc (lưới, đường, cây) từ đồ thị" ⇒ nhìn **bậc** để tìm các điểm đặc biệt (góc, lá).
- Khi một bước có hai lựa chọn đối xứng, chọn bất kỳ (lưới có 8 phép đối xứng); khi lựa chọn **mơ hồ** (lưới 2 hàng), đổi điểm xuất phát/hướng để hết mơ hồ.
- Bài nhiều đáp án ⇒ viết `is_valid`.

## 15. Biến thể của bài

1. Kiểm tra một đồ thị **có phải** lưới không (trả về lưới hoặc `[]`).
2. Lưới **xuyến** (torus, mọi nút bậc 4) — không còn góc, cần cách khác (chọn một nút, tìm chu trình độ dài 4).
3. Lưới tam giác/lục giác.
4. Tìm kích thước `(r, c)` mà không dựng lưới: từ số nút bậc 1/2/3/4.

In [16]:
# Biến thể 1 và 4
def try_grid(n, edges):
    try:
        gr = Solution().constructGridLayout(n, edges)
    except (StopIteration, ValueError, IndexError):
        return []
    return gr if is_valid(n, edges, gr) else []

def shape_from_degrees(n, edges):
    deg = [0] * n
    for u, v in edges:
        deg[u] += 1; deg[v] += 1
    if min(deg) == 1:
        return (1, n)
    b3 = deg.count(3)                       # số nút biên không phải góc = 2(r-2) + 2(c-2)
    s = b3 // 2 + 4                          # r + c
    for r in range(2, s):
        c = s - r
        if r <= c and r * c == n:
            return (r, c)
    return None

rng = random.Random(15)
for _ in range(200):
    r, c = rng.randint(1, 7), rng.randint(1, 7)
    if r * c < 2:
        continue
    n, edges, _ = make_grid_graph(r, c, rng)
    assert try_grid(n, edges)
    assert shape_from_degrees(n, edges) == (min(r, c), max(r, c))
cycle5 = [[i, (i + 1) % 5] for i in range(5)]
assert try_grid(5, cycle5) == []            # chu trình 5 không phải lưới
print('Biến thể 1 (nhận dạng) và 4 (kích thước từ bậc) đúng.')

Biến thể 1 (nhận dạng) và 4 (kích thước từ bậc) đúng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi hoán vị | `O(n! n)` | Oracle |
| 2 | Hàng đầu + mở rộng | `O(n + E)` | **Nộp bài** |
| 3 | Tọa độ từ BFS | `O(n + E)` | Đẹp, dùng tính chất Manhattan |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 2`, `edges = [[0, 1]]`.

**Bài 2 (tính tay):** với lưới `3 × 5`, có bao nhiêu nút bậc 2, 3, 4?

**Bài 3 (điền TODO):** trả về lưới đã **chuẩn hóa**: góc trên-trái là nút góc có số hiệu nhỏ nhất và hàng đầu đi về phía hàng xóm có số hiệu nhỏ hơn.

**Bài 4 (tự viết lại):** dựng lưới bằng cách 3 nhưng **không** cần biết trước góc nào gần nhất (thử cả ba góc còn lại).

**Bài 5 (mini challenge):** đếm số lưới hợp lệ khác nhau (theo các phép đối xứng) cho một đồ thị lưới `r × c`.

In [17]:
def canonical_todo(n, edges):
    # TODO
    raise NotImplementedError

def by_distances_any_corner_todo(n, edges):
    # TODO
    raise NotImplementedError

def count_layouts_todo(r, c):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường thẳng 2 nút. (2) Bậc 1 cả hai. (3) `[[0, 1]]` hoặc `[[1, 0]]` (hoặc dạng cột).

**Bài 2.** (1) Góc: 4. (2) Biên: `2(3−2) + 2(5−2) = 8`. (3) Trong: `1 · 3 = 3`.

**Bài 3.** (1) Lấy một lưới bất kỳ. (2) Thử 8 phép đối xứng (4 nếu không vuông + chuyển vị). (3) Chọn bản thỏa điều kiện.

**Bài 4.** (1) Với mỗi góc `B ≠ A`, thử công thức. (2) Kiểm `is_valid`. (3) Trả bản hợp lệ đầu tiên.

**Bài 5.** (1) Mỗi lưới hợp lệ là một phép đối xứng của lưới gốc. (2) Hình chữ nhật không vuông: 4 phép (giữ hình dạng) + 4 (chuyển vị) = 8; vuông: 8. (3) Một hàng: 2 (đảo) × 2 (hàng/cột) = 4; `1 × 1` không xét.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2
out = Solution().constructGridLayout(2, [[0, 1]])
assert is_valid(2, [[0, 1]], out)
dm = degree_map(3, 5)
flat = [d for row in dm for d in row]
assert (flat.count(2), flat.count(3), flat.count(4)) == (4, 8, 3)

# Bài 3: chuẩn hóa
def symmetries(grid):
    res = []
    g = grid
    for _ in range(2):
        for _ in range(4):
            res.append(g)
            g = [list(r) for r in zip(*g[::-1])]           # xoay 90°
        g = [row[::-1] for row in grid]                     # lật
    return res

def canonical(n, edges):
    base = Solution().constructGridLayout(n, edges)
    deg = [0] * n
    for u, v in edges:
        deg[u] += 1; deg[v] += 1
    cand = [s for s in symmetries(base)]
    m = min(s[0][0] for s in cand)
    cand = [s for s in cand if s[0][0] == m]
    def key(s):
        right = s[0][1] if len(s[0]) > 1 else float('inf')
        return (right, len(s))
    return min(cand, key=key)

rng = random.Random(3)
for _ in range(100):
    r, c = rng.randint(1, 6), rng.randint(1, 6)
    if r * c < 2:
        continue
    n, edges, _ = make_grid_graph(r, c, rng)
    cg = canonical(n, edges)
    assert is_valid(n, edges, cg)
print('Bài 1, 2 đúng | Bài 3: lưới chuẩn hóa luôn hợp lệ.')

Bài 1, 2 đúng | Bài 3: lưới chuẩn hóa luôn hợp lệ.


In [19]:
# Bài 4, 5
def by_distances_any_corner(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    deg = [len(a) for a in g]
    corners = [v for v in range(n) if deg[v] == min(deg)]
    A = corners[0]; dA = bfs(g, A)
    for B in corners[1:]:
        dB = bfs(g, B); c = dA[B] + 1
        if n % c:
            continue
        r = n // c
        grid = [[-1] * c for _ in range(r)]
        ok = True
        for v in range(n):
            i2, j2 = dA[v] + dB[v] - (c - 1), dA[v] - dB[v] + c - 1
            if i2 % 2 or j2 % 2 or not (0 <= i2 // 2 < r and 0 <= j2 // 2 < c) or grid[i2 // 2][j2 // 2] != -1:
                ok = False; break
            grid[i2 // 2][j2 // 2] = v
        if ok and is_valid(n, edges, grid):
            return grid
    return []

def count_layouts(r, c):
    n, edges, grid = make_grid_graph(r, c, random.Random(0))
    return len({tuple(map(tuple, s)) for s in symmetries(grid)})

rng = random.Random(4)
for _ in range(150):
    r, c = rng.randint(1, 6), rng.randint(1, 6)
    if r * c < 2:
        continue
    n, edges, _ = make_grid_graph(r, c, rng)
    assert is_valid(n, edges, by_distances_any_corner(n, edges))
assert count_layouts(3, 4) == 8 and count_layouts(3, 3) == 8 and count_layouts(1, 5) == 4
print('Bài 4 hợp lệ; Bài 5: 3×4 -> 8, 3×3 -> 8, 1×5 -> 4 cách bố trí.')

Bài 4 hợp lệ; Bài 5: 3×4 -> 8, 3×3 -> 8, 1×5 -> 4 cách bố trí.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
x = nút bậc nhỏ nhất
if deg[x] == 1: đi dọc đường thẳng
else:
    row = [x, hàng xóm bậc nhỏ nhất của x]
    while deg[row[-1]] != 2: row.append(hàng xóm chưa dùng có bậc < 4)
    while còn nút: hàng mới = [hàng xóm chưa dùng duy nhất của u] for u in hàng trước
```

**Checklist:**

- [ ] Xử lý lưới một hàng (bậc 1)?
- [ ] Lưới 2 hàng: chọn hàng xóm bậc nhỏ nhất của góc?
- [ ] Đi biên bằng bộ lọc bậc `< 4`?
- [ ] Đánh dấu `used` ngay khi đặt nút?
- [ ] Test bằng `is_valid` (nhiều đáp án)?